In [1]:
import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam, Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

import time

c:\Users\kgl07\anaconda3\envs\course42186\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set random seed for reproducibility
pyro.set_rng_seed(42)

In [3]:
# load the data
test_data_path = "small_test_data.csv"
test_label_path = "small_test_labels.csv"
test_data = pd.read_csv(test_data_path).values.astype(np.float32)
test_label = pd.read_csv(test_label_path).values

val_data_path = "small_val_data.csv"
val_label_path = "small_val_labels.csv"
val_data = pd.read_csv(val_data_path).values.astype(np.float32)
val_label = pd.read_csv(val_label_path).values

print("test_data shape:", test_data.shape)
print("test_label shape:", test_label.shape)

print(test_data[0,0:10])

test_data shape: (200, 400)
test_label shape: (200, 1)
[5.53000e-02 0.00000e+00 5.55492e+01 4.71580e+00 3.67360e+00 2.70879e+01
 1.88500e-01 1.50104e+01 1.45420e+01 1.54886e+01]


In [30]:
print(f"AML", np.sum(test_label == 0))
print(f"ALL", np.sum(test_label == 1))
print(f"Normal", np.sum(test_label == 2))

AML 118
ALL 67
Normal 15


In [4]:
X_train = test_data


# Z-score normalization per gene
X_train_mean = X_train.mean(axis=0)
X_train_std = X_train.std(axis=0) + 1e-6  # avoid div-by-zero
X_norm = (X_train - X_train_mean) / X_train_std
X_clipped = np.clip(X_norm, -5, 5)

X_train_tensor = torch.tensor(X_clipped, dtype=torch.float32)


print(X_train_tensor[0,0:10])

tensor([-0.3517, -0.2533, -0.2346, -0.3010, -0.4803, -0.4995, -0.2547, -0.1780,
         0.0544, -1.0121])


In [ ]:
def supervised_ppca_model(X, y, latent_dim, n_classes):
    n_samples, n_genes = X.shape

    # Sample W from prior
    W = pyro.sample("W", dist.Normal(0., 1.).expand([latent_dim, n_genes]).to_event(2))

    # Sigma is still treated as a point estimate for simplicity
    sigma = pyro.param("sigma_loc", torch.tensor(1.0), constraint=dist.constraints.positive)

    # Classifier weights (point estimate)
    A = pyro.param("A", torch.randn(n_classes, latent_dim))
    b = pyro.param("b", torch.zeros(n_classes))

    with pyro.plate("data", n_samples):
        # Sample latent variable
        z = pyro.sample("z", dist.Normal(0., 1.).expand([latent_dim]).to_event(1))

        # Likelihood for gene expression
        x_loc = torch.matmul(z, W)
        pyro.sample("obs", dist.Normal(x_loc, sigma).to_event(1), obs=X)

        # Likelihood for class label
        probs = torch.sigmoid(torch.matmul(z, A.T) + b)
        pyro.sample("y", dist.Categorical(probs=probs), obs=y)


In [6]:
def supervised_ppca_guide(X, y, latent_dim, n_classes):
    n_samples, n_genes = X.shape


    # Variational posterior for W
    w_loc = pyro.param("w_loc", torch.randn(latent_dim, n_genes))
    w_scale = pyro.param("w_scale", torch.ones(latent_dim, n_genes), constraint=dist.constraints.positive)
    pyro.sample("W", dist.Normal(w_loc, w_scale).to_event(2))

    # Variational posterior for z
    z_loc = pyro.param("z_loc", torch.randn(n_samples, latent_dim))
    z_scale = pyro.param("z_scale", torch.ones(n_samples, latent_dim), constraint=dist.constraints.positive)
    
    with pyro.plate("data", n_samples):
        pyro.sample("z", dist.Normal(z_loc, z_scale).to_event(1))


In [11]:
X_train_tensor = torch.tensor(X_clipped, dtype=torch.float32)
y_train_tensor = torch.tensor(test_label, dtype=torch.long).flatten()
latent_dim = 20
n_classes = 3

In [20]:
pyro.clear_param_store()
optimizer = ClippedAdam({"lr": 0.01})
svi = SVI(supervised_ppca_model, supervised_ppca_guide, optimizer, loss=Trace_ELBO())

start_time = time.time()

n_steps = 3000
for step in range(n_steps):
    loss = svi.step(X_train_tensor, y_train_tensor, latent_dim, n_classes)
    if step % 200 == 0:
        print(f"[Step {step}] Loss: {loss:.2f}")

# Total time spent
print(f"Total time spent: {time.time() - start_time:.2f} seconds")
print(f"Total time spent: {((time.time() - start_time) / 60):.2f} minutes")

[Step 0] Loss: 3329741.97
[Step 200] Loss: 146688.05
[Step 400] Loss: 105084.50
[Step 600] Loss: 92568.23
[Step 800] Loss: 88694.14
[Step 1000] Loss: 87854.94
[Step 1200] Loss: 87712.00
[Step 1400] Loss: 87654.35
[Step 1600] Loss: 87574.76
[Step 1800] Loss: 87407.01
[Step 2000] Loss: 87499.03
[Step 2200] Loss: 87348.57
[Step 2400] Loss: 87334.75
[Step 2600] Loss: 87251.66
[Step 2800] Loss: 87275.51
Total time spent: 21.97 seconds
Total time spent: 0.37 minutes


In [ ]:
W_post = pyro.param("w_loc")  # shape: [latent_dim, n_genes]
A = pyro.param("A")           # shape: [n_classes, latent_dim]
b = pyro.param("b")           # shape: [n_classes]

In [22]:
def infer_z_val_model(X_val_tensor, latent_dim):
    n_val = X_val_tensor.shape[0]
    W = pyro.param("w_loc")                # Use learned W
    sigma = pyro.param("sigma_loc")        # Fixed sigma

    with pyro.plate("data", n_val):
        z = pyro.sample("z", dist.Normal(0., 1.).expand([latent_dim]).to_event(1))
        x_loc = torch.matmul(z, W)
        pyro.sample("obs", dist.Normal(x_loc, sigma).to_event(1), obs=X_val_tensor)


def infer_z_val_guide(X_val_tensor, latent_dim):
    n_val = X_val_tensor.shape[0]

    # New variational parameters for validation latent space
    z_loc_val = pyro.param("z_loc_val", torch.randn(n_val, latent_dim))
    z_scale_val = pyro.param("z_scale_val", torch.ones(n_val, latent_dim), constraint=dist.constraints.positive)

    with pyro.plate("data", n_val):
        pyro.sample("z", dist.Normal(z_loc_val, z_scale_val).to_event(1))


In [23]:
X_val_tensor = torch.tensor(val_data, dtype=torch.float32)
y_val = torch.tensor(val_label, dtype=torch.float32).numpy()

In [24]:
optimizer = ClippedAdam({"lr": 0.01})
svi_val = SVI(infer_z_val_model, infer_z_val_guide, optimizer, loss=Trace_ELBO())

start_time = time.time()

n_steps = 3000
for step in range(n_steps):
    loss = svi_val.step(X_val_tensor, latent_dim)
    if step % 200 == 0:
        print(f"[Step {step}] Loss: {loss:.2f}")

# Total time spent
print(f"Total time spent: {time.time() - start_time:.2f} seconds")
print(f"Total time spent: {((time.time() - start_time) / 60):.2f} minutes")

[Step 0] Loss: 17299109858.46
[Step 200] Loss: 308334713.33
[Step 400] Loss: 5729091.90
[Step 600] Loss: 627821.78
[Step 800] Loss: 597122.07
[Step 1000] Loss: 590119.47
[Step 1200] Loss: 586045.47
[Step 1400] Loss: 583876.77
[Step 1600] Loss: 582795.04
[Step 1800] Loss: 582103.07
[Step 2000] Loss: 581513.47
[Step 2200] Loss: 580936.60
[Step 2400] Loss: 580330.35
[Step 2600] Loss: 579644.93
[Step 2800] Loss: 578875.79
Total time spent: 12.94 seconds
Total time spent: 0.22 minutes


In [25]:
z_loc_val = pyro.param("z_loc_val")  # [n_val, latent_dim]

# Predict class logits
logits_val = torch.matmul(z_loc_val, A.T) + b
y_pred_val = torch.argmax(logits_val, dim=1)

accuracy = (y_pred_val == y_val).float().mean().item()
print(f"Validation accuracy: {accuracy:.4f}")


Validation accuracy: 0.5419


In [33]:
print(f"AML", np.sum(val_label == 0))
print(f"ALL", np.sum(val_label == 1))
print(f"Normal", np.sum(val_label == 2))

y_pred_val = y_pred_val.numpy()
print(f"AML", np.sum(y_pred_val == 0))
print(f"ALL", np.sum(y_pred_val == 1))
print(f"Normal", np.sum(y_pred_val == 2))

AML 118
ALL 68
Normal 14
AML 181
ALL 1
Normal 18
